<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: OpenMDW-1.1 -->

# Cosmos3 PAI-Bench Reproduction with Cosmos Framework

This notebook walks through generating the [PAI-Bench (Physical AI Bench) generation set](https://huggingface.co/datasets/shi-labs/physical-ai-bench-generation) with Cosmos3-Super using the native Cosmos Framework PyTorch entrypoint:

```bash
python -m cosmos_framework.scripts.inference
```

PAI-Bench covers Physical AI domains (AV driving, robotics, industry, physics, human, common sense) with **1044 samples**. This notebook runs **both** generation tasks:

- **Text-to-Video (T2V):** generate from the prompt only (no condition image).
- **Image-to-Video (I2V):** condition on the per-sample image (`condition_image/<image_name>`) and generate.

Both tasks use the local prompt assets under `assets/` (which carry `json_upsampled_prompt` and `negative_prompt`):

- `assets/t2v_prompts.json`
- `assets/i2v_prompts.json`

Generation target: **189 frames at 24 FPS**. We walk through one T2V demo and one I2V demo end-to-end, then provide two optional full-sweep cells (one per task).

## Prerequisites

- Linux machine with NVIDIA GPU access (default recipe uses 4 GPUs).
- Model access on Hugging Face. Either run `uvx hf@latest auth login` or set `HF_TOKEN` in the environment.
- `uv >= 0.11.3` installed (https://docs.astral.sh/uv/getting-started/installation/).
- `git-lfs` on PATH. The PAI-Bench dataset is downloaded by cloning the Hugging Face dataset repo (images are LFS-backed).
- Cache/output paths with enough disk space.

> **Headless servers:** if you see `libxcb.so.1: cannot open shared object file` when importing the model, install the system graphics libraries:
> ```bash
> apt-get install -y libxcb1 libgl1 libglib2.0-0
> ```

## 1. Configure Paths and Environment

All paths default to sensible locations under this `cosmos` checkout. Override any of them by exporting before launching the notebook:

```bash
export COSMOS3_REPO=/path/to/cosmos-framework
export COSMOS3_UV_GROUP=cu130-train   # or cu128-train
export UV_PROJECT_ENVIRONMENT=/path/to/large/uv/venvs/cosmos3-paibench
export COSMOS3_NUM_GPUS=4
export HF_HOME=/path/to/large/huggingface/cache
export CUDA_VISIBLE_DEVICES=0,1,2,3
export PAIBENCH_DATASET_ROOT=/path/to/physical-ai-bench-generation
export PAIBENCH_OUTPUT_ROOT=/path/to/paibench/outputs
```

In [ ]:
from pathlib import Path
import os
import socket


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "README.md").exists() and (path / "cookbooks").exists():
            return path
    return start


def free_local_port() -> str:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("127.0.0.1", 0))
        return str(sock.getsockname()[1])


def default_framework_repo(root: Path) -> Path:
    for candidate in (root / "packages" / "cosmos-framework", root / "packages" / "cosmos3"):
        if (candidate / "pyproject.toml").exists() and (candidate / "cosmos_framework").exists():
            return candidate
    return root / "packages" / "cosmos-framework"


COSMOS_ROOT = find_repo_root(Path.cwd().resolve())
COSMOS3_REPO = Path(os.environ.get("COSMOS3_REPO", default_framework_repo(COSMOS_ROOT))).resolve()
COSMOS3_GIT_URL = os.environ.get("COSMOS3_GIT_URL", "git@github.com:NVIDIA/cosmos-framework.git")
COSMOS3_UV_GROUP = os.environ.get("COSMOS3_UV_GROUP", "cu130-train")
COSMOS3_UV_ENV = Path(os.environ.get("UV_PROJECT_ENVIRONMENT", COSMOS3_REPO / ".venv")).resolve()
COSMOS3_NUM_GPUS = os.environ.get("COSMOS3_NUM_GPUS", "4")
CUDA_VISIBLE_DEVICES = os.environ.get("CUDA_VISIBLE_DEVICES", "0,1,2,3")

PAIBENCH_NOTEBOOK_ROOT = COSMOS_ROOT / "evaluation" / "paibench_g"
PAIBENCH_ASSETS = PAIBENCH_NOTEBOOK_ROOT / "assets"
I2V_PROMPTS_FILE = PAIBENCH_ASSETS / "i2v_prompts.json"
T2V_PROMPTS_FILE = PAIBENCH_ASSETS / "t2v_prompts.json"

PAIBENCH_HF_URL = os.environ.get(
    "PAIBENCH_HF_URL", "https://huggingface.co/datasets/shi-labs/physical-ai-bench-generation"
)
PAIBENCH_DATASET_ROOT = Path(
    os.environ.get("PAIBENCH_DATASET_ROOT", PAIBENCH_NOTEBOOK_ROOT / "physical-ai-bench-generation")
).resolve()
PAIBENCH_CONDITION_IMAGE_DIR = PAIBENCH_DATASET_ROOT / "condition_image"
PAIBENCH_OUTPUT_ROOT = Path(
    os.environ.get("PAIBENCH_OUTPUT_ROOT", PAIBENCH_NOTEBOOK_ROOT / "outputs")
).resolve()
PAIBENCH_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DEMO_CASE_ID = os.environ.get("PAIBENCH_DEMO_CASE", "av_6a0c672b-9603-410e-bb4b-323c7c1b8029")
CHECKPOINT = os.environ.get("PAIBENCH_CHECKPOINT", "Cosmos3-Super")

MASTER_ADDR = os.environ.get("COSMOS3_MASTER_ADDR", "127.0.0.1")
MASTER_PORT_I2V = os.environ.get("COSMOS3_I2V_MASTER_PORT", free_local_port())
MASTER_PORT_T2V = os.environ.get("COSMOS3_T2V_MASTER_PORT", free_local_port())

for key, value in [
    ("COSMOS_ROOT", COSMOS_ROOT),
    ("COSMOS3_REPO", COSMOS3_REPO),
    ("COSMOS3_UV_ENV", COSMOS3_UV_ENV),
    ("COSMOS3_NUM_GPUS", COSMOS3_NUM_GPUS),
    ("CUDA_VISIBLE_DEVICES", CUDA_VISIBLE_DEVICES),
    ("PAIBENCH_ASSETS", PAIBENCH_ASSETS),
    ("PAIBENCH_DATASET_ROOT", PAIBENCH_DATASET_ROOT),
    ("PAIBENCH_OUTPUT_ROOT", PAIBENCH_OUTPUT_ROOT),
    ("DEMO_CASE_ID", DEMO_CASE_ID),
    ("CHECKPOINT", CHECKPOINT),
]:
    print(f"{key}={value}")

# Export for the %%bash cells below.
os.environ["COSMOS3_REPO"] = str(COSMOS3_REPO)
os.environ["COSMOS3_GIT_URL"] = COSMOS3_GIT_URL
os.environ["COSMOS3_UV_GROUP"] = COSMOS3_UV_GROUP
os.environ["COSMOS3_UV_ENV"] = str(COSMOS3_UV_ENV)
os.environ["UV_PROJECT_ENVIRONMENT"] = str(COSMOS3_UV_ENV)
os.environ["COSMOS3_NUM_GPUS"] = COSMOS3_NUM_GPUS
os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES
os.environ["PAIBENCH_HF_URL"] = PAIBENCH_HF_URL
os.environ["PAIBENCH_DATASET_ROOT"] = str(PAIBENCH_DATASET_ROOT)
os.environ["PAIBENCH_OUTPUT_ROOT"] = str(PAIBENCH_OUTPUT_ROOT)
os.environ["DEMO_CASE_ID"] = DEMO_CASE_ID
os.environ["CHECKPOINT"] = CHECKPOINT
os.environ["COSMOS3_MASTER_ADDR"] = MASTER_ADDR
os.environ["COSMOS3_I2V_MASTER_PORT"] = MASTER_PORT_I2V
os.environ["COSMOS3_T2V_MASTER_PORT"] = MASTER_PORT_T2V

## 2. Clone or Reuse Cosmos Framework

In [ ]:
%%bash
set -euo pipefail

mkdir -p "$(dirname "$COSMOS3_REPO")"

if [ -f "$COSMOS3_REPO/pyproject.toml" ] && [ -d "$COSMOS3_REPO/cosmos_framework" ]; then
  echo "Using existing framework checkout: $COSMOS3_REPO"
elif [ -e "$COSMOS3_REPO" ]; then
  echo "COSMOS3_REPO exists but is not a Cosmos Framework checkout: $COSMOS3_REPO"
  exit 1
else
  echo "Cloning $COSMOS3_GIT_URL into $COSMOS3_REPO"
  git clone "$COSMOS3_GIT_URL" "$COSMOS3_REPO"
fi

cd "$COSMOS3_REPO"
git status --short --branch
git remote -v

## 3. Install Native PyTorch Dependencies

Installs framework dependencies with the requested CUDA group (default `cu130-train`).

In [ ]:
%%bash
set -euo pipefail

if ! command -v uv >/dev/null 2>&1; then
  echo "uv is not installed. Install it first: https://docs.astral.sh/uv/getting-started/installation/"
  exit 1
fi

export GIT_LFS_SKIP_SMUDGE=1
cd "$COSMOS3_REPO"
export UV_PROJECT_ENVIRONMENT="${UV_PROJECT_ENVIRONMENT:-$COSMOS3_UV_ENV}"
echo "Using UV_PROJECT_ENVIRONMENT=$UV_PROJECT_ENVIRONMENT"
uv sync --all-extras --group="$COSMOS3_UV_GROUP"
if [ ! -x "$COSMOS3_UV_ENV/bin/python" ]; then
  echo "uv sync completed, but expected Python is missing: $COSMOS3_UV_ENV/bin/python"
  exit 1
fi

## 4. Verify GPU and Python Environment

In [ ]:
%%bash
set -euo pipefail

cd "$COSMOS3_REPO"
if [ ! -x "$COSMOS3_UV_ENV/bin/python" ]; then
  echo "Missing $COSMOS3_UV_ENV/bin/python"
  echo "Run the Install Native PyTorch Dependencies cell first."
  exit 1
fi
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" "$COSMOS3_UV_ENV/bin/python" - <<'PY'
import torch
print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(f"device {index}:", torch.cuda.get_device_name(index))
PY

## 5. Download the PAI-Bench Dataset

PAI-Bench is hosted on Hugging Face at [`shi-labs/physical-ai-bench-generation`](https://huggingface.co/datasets/shi-labs/physical-ai-bench-generation). We download it by cloning the dataset repo with Git LFS.

Layout (under `$PAIBENCH_DATASET_ROOT`):

```
physical-ai-bench-generation/
├── condition_image/                       # condition images (filenames match each sample's image_name)
├── vqa/                                   # VQA pairs (not used here)
└── cosmos_predict2_bench_full_info.json   # dataset metadata
```

> Only the condition images are read from the cloned dataset (for I2V). The prompts come from the **local** assets at `assets/i2v_prompts.json` and `assets/t2v_prompts.json` (which include `json_upsampled_prompt` and `negative_prompt`).

In [ ]:
%%bash
set -euo pipefail

if [ -d "$PAIBENCH_DATASET_ROOT/condition_image" ]; then
  echo "PAI-Bench dataset already present at $PAIBENCH_DATASET_ROOT"
  ls "$PAIBENCH_DATASET_ROOT"
  exit 0
fi

if ! command -v git-lfs >/dev/null 2>&1; then
  echo "git-lfs is required to download the PAI-Bench dataset (LFS-backed images)." >&2
  echo "Install it: https://git-lfs.com/  (e.g. apt-get install -y git-lfs)" >&2
  exit 1
fi

git lfs install
mkdir -p "$(dirname "$PAIBENCH_DATASET_ROOT")"
git clone "$PAIBENCH_HF_URL" "$PAIBENCH_DATASET_ROOT"

echo "--- contents of $PAIBENCH_DATASET_ROOT ---"
ls "$PAIBENCH_DATASET_ROOT"

## 6. Load Prompts and Preview the Demo Case

We load the T2V and I2V prompt assets, each keyed by `video_id` (1044 entries each). For I2V, the condition image for a case is `condition_image/<image_name>` inside the cloned dataset.

In [ ]:
import json
from IPython.display import Image, display

I2V_PROMPTS = {row["video_id"]: row for row in json.loads(I2V_PROMPTS_FILE.read_text())}
T2V_PROMPTS = {row["video_id"]: row for row in json.loads(T2V_PROMPTS_FILE.read_text())}

print(f"Loaded {len(I2V_PROMPTS)} I2V prompts from {I2V_PROMPTS_FILE.name}")
print(f"Loaded {len(T2V_PROMPTS)} T2V prompts from {T2V_PROMPTS_FILE.name}")
assert len(I2V_PROMPTS) == 1044, f"expected 1044 I2V prompts, got {len(I2V_PROMPTS)}"
assert len(T2V_PROMPTS) == 1044, f"expected 1044 T2V prompts, got {len(T2V_PROMPTS)}"

assert DEMO_CASE_ID in I2V_PROMPTS, f"{DEMO_CASE_ID} not found in I2V prompts"
assert DEMO_CASE_ID in T2V_PROMPTS, f"{DEMO_CASE_ID} not found in T2V prompts"

entry = I2V_PROMPTS[DEMO_CASE_ID]
demo_image = PAIBENCH_CONDITION_IMAGE_DIR / entry["image_name"]
print("\ndemo case:", DEMO_CASE_ID)
print("condition image (I2V):", demo_image)
print("prompt_en:", entry.get("prompt_en", "")[:300], "...")

if demo_image.exists():
    display(Image(filename=str(demo_image), width=480))
else:
    print("Condition image not found yet - run the dataset download cell first.")

## 7. Helper Functions

**Recipe (shared by T2V and I2V)**

- **Output length**: 189 frames at 24 fps.
- **Sampling**: `num_steps=50`, `guidance=6.0`, `shift=10.0`, `seed=0` (all other sampler settings left at framework defaults).
- **Positive prompt**: the per-case `json_upsampled_prompt` string, passed through verbatim.
- **Negative prompt**: the shared `negative_prompt` string, passed through verbatim.
- **T2V**: `model_mode="text2video"`, no condition image.
- **I2V**: `model_mode="image2video"`, conditioned on `condition_image/<image_name>`.

Helpers:

- `case_image_path(case_id)` — resolve the I2V condition image (falls back to a recursive search if the clone nests the images).
- `build_t2v_row(case_id)` / `build_i2v_row(case_id)` — assemble inference JSONL rows.
- `build_input_jsonl(rows, dst)` — write one or more rows to a JSONL file.

In [ ]:
NUM_FRAMES = 189
FPS = 24
RESOLUTION = "720"
ASPECT_RATIO = "16,9"
NUM_STEPS = 50
GUIDANCE = 6.0
SHIFT = 10.0
SEED = 0


def case_image_path(case_id: str) -> Path:
    entry = I2V_PROMPTS[case_id]
    image_name = entry["image_name"]
    direct = PAIBENCH_CONDITION_IMAGE_DIR / image_name
    if direct.exists():
        return direct
    # Fall back to a recursive search in case the clone nests the images differently.
    matches = sorted(PAIBENCH_DATASET_ROOT.rglob(image_name))
    if matches:
        return matches[0]
    raise FileNotFoundError(f"condition image not found for {case_id}: {direct}")


def build_t2v_row(case_id: str) -> dict:
    entry = T2V_PROMPTS[case_id]
    return {
        "aspect_ratio": ASPECT_RATIO,
        "fps": FPS,
        "guidance": GUIDANCE,
        "model_mode": "text2video",
        "name": case_id,
        "negative_prompt": entry["negative_prompt"],
        "num_frames": NUM_FRAMES,
        "num_outputs": 1,
        "num_steps": NUM_STEPS,
        "prompt": entry["json_upsampled_prompt"],
        "resolution": RESOLUTION,
        "seed": SEED,
        "shift": SHIFT,
    }


def build_i2v_row(case_id: str) -> dict:
    entry = I2V_PROMPTS[case_id]
    return {
        "aspect_ratio": ASPECT_RATIO,
        "fps": FPS,
        "guidance": GUIDANCE,
        "model_mode": "image2video",
        "name": case_id,
        "negative_prompt": entry["negative_prompt"],
        "num_frames": NUM_FRAMES,
        "num_outputs": 1,
        "num_steps": NUM_STEPS,
        "prompt": entry["json_upsampled_prompt"],
        "resolution": RESOLUTION,
        "seed": SEED,
        "shift": SHIFT,
        "vision_path": str(case_image_path(case_id)),
    }


def build_input_jsonl(rows: list[dict], dst_jsonl: Path) -> Path:
    dst_jsonl.parent.mkdir(parents=True, exist_ok=True)
    with dst_jsonl.open("w") as fp:
        for row in rows:
            fp.write(json.dumps(row) + "\n")
    return dst_jsonl


def display_video(path: Path, width: int = 480) -> None:
    from IPython.display import Video, display
    display(Video(filename=str(path), embed=True, width=width))

## 8. I2V Demo — Build, Run, Preview

In [ ]:
i2v_run_dir = PAIBENCH_OUTPUT_ROOT / "i2v" / DEMO_CASE_ID
i2v_run_dir.mkdir(parents=True, exist_ok=True)

i2v_input_jsonl = i2v_run_dir / "input.jsonl"
i2v_output_dir = i2v_run_dir / "raw"
i2v_output_dir.mkdir(parents=True, exist_ok=True)

build_input_jsonl([build_i2v_row(DEMO_CASE_ID)], i2v_input_jsonl)

os.environ["I2V_INPUT"] = str(i2v_input_jsonl)
os.environ["I2V_OUTPUT_DIR"] = str(i2v_output_dir)

print("I2V_INPUT =", i2v_input_jsonl)
print("I2V_OUTPUT_DIR =", i2v_output_dir)
print()
print("row preview:")
print(i2v_input_jsonl.read_text()[:600], "...")

### Run I2V Inference

We use the **latency** parallelism preset with context-parallel sharding across all visible GPUs (`--cp-size=$COSMOS3_NUM_GPUS`).

In [ ]:
%%bash
set -euo pipefail

cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
"$COSMOS3_UV_ENV/bin/torchrun" \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  --master-addr="$COSMOS3_MASTER_ADDR" \
  --master-port="$COSMOS3_I2V_MASTER_PORT" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  --dp-shard-size=1 --dp-replicate-size=1 \
  --cp-size="$COSMOS3_NUM_GPUS" --cfgp-size=1 \
  -i "$I2V_INPUT" \
  -o "$I2V_OUTPUT_DIR" \
  --checkpoint-path "$CHECKPOINT" \
  --no-guardrails

In [ ]:
raw_outputs = sorted(i2v_output_dir.rglob("vision.mp4"))
if raw_outputs:
    raw_mp4 = raw_outputs[0]
    print("raw I2V output:", raw_mp4)
    display_video(raw_mp4)
else:
    print("No vision.mp4 found yet - run the I2V inference cell first.")

## 9. T2V Demo — Build, Run, Preview

Text-to-Video uses the same case prompt but no condition image.

In [ ]:
t2v_run_dir = PAIBENCH_OUTPUT_ROOT / "t2v" / DEMO_CASE_ID
t2v_run_dir.mkdir(parents=True, exist_ok=True)

t2v_input_jsonl = t2v_run_dir / "input.jsonl"
t2v_output_dir = t2v_run_dir / "raw"
t2v_output_dir.mkdir(parents=True, exist_ok=True)

build_input_jsonl([build_t2v_row(DEMO_CASE_ID)], t2v_input_jsonl)

os.environ["T2V_INPUT"] = str(t2v_input_jsonl)
os.environ["T2V_OUTPUT_DIR"] = str(t2v_output_dir)

print("T2V_INPUT =", t2v_input_jsonl)
print("T2V_OUTPUT_DIR =", t2v_output_dir)
print()
print("row preview:")
print(t2v_input_jsonl.read_text()[:600], "...")

### Run T2V Inference

In [ ]:
%%bash
set -euo pipefail

cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
"$COSMOS3_UV_ENV/bin/torchrun" \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  --master-addr="$COSMOS3_MASTER_ADDR" \
  --master-port="$COSMOS3_T2V_MASTER_PORT" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  --dp-shard-size=1 --dp-replicate-size=1 \
  --cp-size="$COSMOS3_NUM_GPUS" --cfgp-size=1 \
  -i "$T2V_INPUT" \
  -o "$T2V_OUTPUT_DIR" \
  --checkpoint-path "$CHECKPOINT" \
  --no-guardrails

In [ ]:
raw_outputs = sorted(t2v_output_dir.rglob("vision.mp4"))
if raw_outputs:
    raw_mp4 = raw_outputs[0]
    print("raw T2V output:", raw_mp4)
    display_video(raw_mp4)
else:
    print("No vision.mp4 found yet - run the T2V inference cell first.")

## 10. (Optional) Run All 1044 Cases — I2V

Set `RUN_ALL_I2V = True` to write a single combined JSONL covering every I2V case, then run the following bash cell to generate them all.

In [ ]:
RUN_ALL_I2V = False  # set to True to enable the full 1044-case I2V sweep

if RUN_ALL_I2V:
    case_ids = sorted(I2V_PROMPTS.keys())
    assert len(case_ids) == 1044

    i2v_inputs_dir = PAIBENCH_OUTPUT_ROOT / "i2v_full" / "inputs"
    i2v_raw_dir = PAIBENCH_OUTPUT_ROOT / "i2v_full" / "raw"
    i2v_inputs_dir.mkdir(parents=True, exist_ok=True)
    i2v_raw_dir.mkdir(parents=True, exist_ok=True)

    all_jsonl = i2v_inputs_dir / "all_1044.jsonl"
    build_input_jsonl([build_i2v_row(case_id) for case_id in case_ids], all_jsonl)

    os.environ["I2V_FULL_INPUT"] = str(all_jsonl)
    os.environ["I2V_FULL_OUTPUT_DIR"] = str(i2v_raw_dir)
    print("I2V_FULL_INPUT =", all_jsonl)
    print("I2V_FULL_OUTPUT_DIR =", i2v_raw_dir)
    print(f"Wrote {len(case_ids)} rows. Run the next bash cell to generate all I2V outputs.")
else:
    print("Set RUN_ALL_I2V = True above and re-run to enable the full 1044-case I2V sweep.")

### Generate All I2V Outputs

In [ ]:
%%bash
set -euo pipefail

if [ -z "${I2V_FULL_INPUT:-}" ]; then
  echo "Set RUN_ALL_I2V = True in the previous Python cell and re-run it first."
  exit 0
fi

cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
"$COSMOS3_UV_ENV/bin/torchrun" \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  --master-addr="$COSMOS3_MASTER_ADDR" \
  --master-port="$COSMOS3_I2V_MASTER_PORT" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  --dp-shard-size=1 --dp-replicate-size=1 \
  --cp-size="$COSMOS3_NUM_GPUS" --cfgp-size=1 \
  -i "$I2V_FULL_INPUT" \
  -o "$I2V_FULL_OUTPUT_DIR" \
  --checkpoint-path "$CHECKPOINT" \
  --no-guardrails

## 11. (Optional) Run All 1044 Cases — T2V

Set `RUN_ALL_T2V = True` to write a single combined JSONL covering every T2V case, then run the following bash cell to generate them all.

In [ ]:
RUN_ALL_T2V = False  # set to True to enable the full 1044-case T2V sweep

if RUN_ALL_T2V:
    case_ids = sorted(T2V_PROMPTS.keys())
    assert len(case_ids) == 1044

    t2v_inputs_dir = PAIBENCH_OUTPUT_ROOT / "t2v_full" / "inputs"
    t2v_raw_dir = PAIBENCH_OUTPUT_ROOT / "t2v_full" / "raw"
    t2v_inputs_dir.mkdir(parents=True, exist_ok=True)
    t2v_raw_dir.mkdir(parents=True, exist_ok=True)

    all_jsonl = t2v_inputs_dir / "all_1044.jsonl"
    build_input_jsonl([build_t2v_row(case_id) for case_id in case_ids], all_jsonl)

    os.environ["T2V_FULL_INPUT"] = str(all_jsonl)
    os.environ["T2V_FULL_OUTPUT_DIR"] = str(t2v_raw_dir)
    print("T2V_FULL_INPUT =", all_jsonl)
    print("T2V_FULL_OUTPUT_DIR =", t2v_raw_dir)
    print(f"Wrote {len(case_ids)} rows. Run the next bash cell to generate all T2V outputs.")
else:
    print("Set RUN_ALL_T2V = True above and re-run to enable the full 1044-case T2V sweep.")

### Generate All T2V Outputs

In [ ]:
%%bash
set -euo pipefail

if [ -z "${T2V_FULL_INPUT:-}" ]; then
  echo "Set RUN_ALL_T2V = True in the previous Python cell and re-run it first."
  exit 0
fi

cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
"$COSMOS3_UV_ENV/bin/torchrun" \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  --master-addr="$COSMOS3_MASTER_ADDR" \
  --master-port="$COSMOS3_T2V_MASTER_PORT" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=latency \
  --dp-shard-size=1 --dp-replicate-size=1 \
  --cp-size="$COSMOS3_NUM_GPUS" --cfgp-size=1 \
  -i "$T2V_FULL_INPUT" \
  -o "$T2V_FULL_OUTPUT_DIR" \
  --checkpoint-path "$CHECKPOINT" \
  --no-guardrails